# Fase 0 — Estructura de mercado (legs) + Order Blocks

Segunda iteración del concepto de swings, después de que la primera versión (ventana fija de N velas) resultó ser ruido — no respetaba la secuencia Higher High/Higher Low ni Lower High/Lower Low, y no tenía relación con movimientos impulsivos reales.

Este notebook usa `trading_ai.structure.swings.detect_structural_swings`: un swing se extiende mientras el precio sigue haciendo nuevos extremos en la misma dirección, y solo se confirma cuando hay una reversión real (retroceso de al menos `ATR_MULT` veces el ATR desde ese extremo). Así la secuencia de swings queda encadenada por construcción, y cada leg corresponde a un movimiento impulsivo real — de donde sale el order block correspondiente (`smartmoneyconcepts.smc.ob`, alimentado con estos swings en vez de los de ventana fija).

Para cada temporalidad configurada en `config/timeframes.yaml`, se generan ventanas que muestran al menos 3 legs consecutivos, con los OBs que caigan dentro marcados.

**Cómo iterar**: ajusta `ATR_MULT` (más alto = legs más grandes/menos frecuentes, más bajo = más sensible) y vuelve a correr.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd().name == "00_rule_spec") else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import yaml
import numpy as np
import pandas as pd
import mplfinance as mpf
from smartmoneyconcepts import smc

from trading_ai.data import store
from trading_ai.structure.swings import detect_structural_swings

Thank you for using SmartMoneyConcepts! ⭐ Please show your support by giving a star on the GitHub repository: https://github.com/joshyattridge/smart-money-concepts


In [2]:
# --- Configuración: ajusta esto entre iteraciones ---
SYMBOL = "EURUSD"
with open(PROJECT_ROOT / "config" / "timeframes.yaml", encoding="utf-8") as f:
    TIMEFRAMES = yaml.safe_load(f)["timeframes"]

ATR_PERIOD = 14
ATR_MULT = 5.0

LEGS_PER_WINDOW = 3  # mínimo de legs consecutivos a mostrar por imagen
WINDOWS_PER_TIMEFRAME = 3  # cuántas ventanas de ejemplo por temporalidad
PAD_BARS_FRACTION = 0.15  # padding a cada lado, como fracción del ancho de la ventana

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "00_rule_spec" / "output" / "market_structure"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def render_structure_window(
    df: pd.DataFrame,
    swing_points: pd.DataFrame,
    obs: pd.DataFrame,
    lo: int,
    hi: int,
    title: str,
    out_path: Path,
) -> None:
    """Renderiza velas[lo:hi] con los swings (legs) y OBs que caigan dentro."""
    window = df.iloc[lo:hi]

    swing_high = pd.Series(index=window.index, dtype=float)
    swing_low = pd.Series(index=window.index, dtype=float)
    window_swings = swing_points[(swing_points.index >= window.index.min()) & (swing_points.index <= window.index.max())]
    for ts, row in window_swings.iterrows():
        if row["HighLow"] == 1:
            swing_high.loc[ts] = row["Level"]
        else:
            swing_low.loc[ts] = row["Level"]

    addplots = [
        mpf.make_addplot(swing_high, type="scatter", marker="v", markersize=100, color="red"),
        mpf.make_addplot(swing_low, type="scatter", marker="^", markersize=100, color="green"),
    ]

    window_obs = obs.iloc[lo:hi].dropna(subset=["OB"])
    for ob_ts, ob_row in window_obs.iterrows():
        ob_pos = df.index.get_loc(ob_ts)
        end_pos = int(ob_row["MitigatedIndex"]) if ob_row["MitigatedIndex"] > 0 else hi - 1
        end_pos = min(end_pos, hi - 1)
        span_idx = df.index[ob_pos : end_pos + 1].intersection(window.index)
        top = pd.Series(index=window.index, dtype=float)
        bottom = pd.Series(index=window.index, dtype=float)
        top.loc[span_idx] = ob_row["Top"]
        bottom.loc[span_idx] = ob_row["Bottom"]
        color = "blue" if ob_row["OB"] == 1 else "darkorange"
        addplots.append(mpf.make_addplot(top, type="line", color=color, width=1.5))
        addplots.append(mpf.make_addplot(bottom, type="line", color=color, width=1.5))

    mpf.plot(
        window,
        type="candle",
        style="yahoo",
        addplot=addplots,
        title=title,
        warn_too_much_data=len(window) + 1,
        savefig=dict(fname=str(out_path), dpi=100),
    )

In [4]:
from datetime import datetime, timezone

n_points_needed = LEGS_PER_WINDOW + 1  # N legs = N+1 swing points consecutivos
summary = []

for timeframe in TIMEFRAMES:
    df = store.get_ohlcv(SYMBOL, timeframe, datetime(2000, 1, 1, tzinfo=timezone.utc), datetime.now(timezone.utc))

    swings = detect_structural_swings(df, atr_period=ATR_PERIOD, atr_mult=ATR_MULT)
    obs = smc.ob(df, swings, close_mitigation=False)
    obs.index = df.index
    swing_points = swings.dropna(subset=["HighLow"])
    swing_positions = [df.index.get_loc(ts) for ts in swing_points.index]

    if len(swing_positions) < n_points_needed:
        print(f"{timeframe}: solo {len(swing_positions)} swings confirmados, no alcanza para {LEGS_PER_WINDOW} legs. Sube ATR_MULT o revisa los datos.")
        continue

    # Grupos de n_points_needed swings consecutivos, elegidos repartidos a lo largo del histórico.
    n_groups = len(swing_positions) - n_points_needed + 1
    group_starts = np.linspace(0, n_groups - 1, num=min(WINDOWS_PER_TIMEFRAME, n_groups), dtype=int)

    for w, g in enumerate(group_starts):
        first_pos = swing_positions[g]
        last_pos = swing_positions[g + n_points_needed - 1]
        span = last_pos - first_pos
        pad = max(3, int(span * PAD_BARS_FRACTION))
        lo = max(0, first_pos - pad)
        hi = min(len(df), last_pos + pad + 1)

        out_path = OUTPUT_DIR / f"{SYMBOL}_{timeframe}_{w:02d}.png"
        title = f"{SYMBOL} {timeframe} — {LEGS_PER_WINDOW} legs — {df.index[first_pos].date()} a {df.index[last_pos].date()}"
        render_structure_window(df, swing_points, obs, lo, hi, title, out_path)
        summary.append({"timeframe": timeframe, "window": w, "bars": hi - lo, "path": str(out_path)})

    print(f"{timeframe}: {len(swing_positions)} swings totales, {len(group_starts)} ventanas renderizadas.")

pd.DataFrame(summary)

1D: 15 swings totales, 3 ventanas renderizadas.


4H: 92 swings totales, 3 ventanas renderizadas.


1H: 461 swings totales, 3 ventanas renderizadas.


M30: 833 swings totales, 3 ventanas renderizadas.


M15: 1595 swings totales, 3 ventanas renderizadas.


M5: 2009 swings totales, 3 ventanas renderizadas.


M1: 2057 swings totales, 3 ventanas renderizadas.


,timeframe,window,bars,path
0,1D,0,180,C:\Users\isaac\Downloads\Projecto de trading I...
1,1D,1,341,C:\Users\isaac\Downloads\Projecto de trading I...
2,1D,2,150,C:\Users\isaac\Downloads\Projecto de trading I...
3,4H,0,221,C:\Users\isaac\Downloads\Projecto de trading I...
4,4H,1,134,C:\Users\isaac\Downloads\Projecto de trading I...
5,4H,2,141,C:\Users\isaac\Downloads\Projecto de trading I...
6,1H,0,290,C:\Users\isaac\Downloads\Projecto de trading I...
7,1H,1,126,C:\Users\isaac\Downloads\Projecto de trading I...
8,1H,2,105,C:\Users\isaac\Downloads\Projecto de trading I...
9,M30,0,468,C:\Users\isaac\Downloads\Projecto de trading I...
